# 03 — Computational Spatial Analysis 運算式空間分析

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewwangarchnycu/gis-open-data-workshop-2026/blob/main/notebooks/03_spatial_analysis.ipynb)

Part of [Mapping the Unknown](https://github.com/andrewwangarchnycu/gis-open-data-workshop-2026) — see [Lesson 04](../lessons/04-qgis-basics/) and [Lesson 05](../lessons/05-computational-gis/).

**Spatial meaning first 先談空間意義**: we repeat, in code, the exact buffer / spatial join / intersection reasoning from the QGIS module — but now reproducible and scriptable. 我們在此以程式碼重現 QGIS 模組中緩衝區／空間 join／交集的推理過程，但這次可重現、可腳本化。

In [ ]:
!pip install geopandas shapely matplotlib contextily -q

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon
import matplotlib.pyplot as plt
import contextily as cx

## Step 1 — Load 載入

In [ ]:
tree_coords = [
    (121.5320, 25.0335), (121.5325, 25.0338), (121.5330, 25.0330),
    (121.5340, 25.0345), (121.5345, 25.0348), (121.5300, 25.0320),
    (121.5305, 25.0322), (121.5360, 25.0360),
]
trees = gpd.GeoDataFrame(
    {"tree_id": range(1, len(tree_coords) + 1)},
    geometry=[Point(xy) for xy in tree_coords],
    crs="EPSG:4326",
)

public_spaces = gpd.GeoDataFrame(
    {"name": ["Riverside Park", "Central Plaza", "Corner Lot Square"]},
    geometry=[
        Polygon([(121.5315, 25.0328), (121.5335, 25.0328), (121.5335, 25.0350), (121.5315, 25.0350)]),
        Polygon([(121.5295, 25.0315), (121.5310, 25.0315), (121.5310, 25.0328), (121.5295, 25.0328)]),
        Polygon([(121.5352, 25.0352), (121.5365, 25.0352), (121.5365, 25.0365), (121.5352, 25.0365)]),
    ],
    crs="EPSG:4326",
)
print(trees.shape, public_spaces.shape)

## Step 2 — Reproject to a metric CRS 重新投影至公尺制 CRS

EPSG:4326 (lon/lat, degrees) cannot be used for a 10-meter buffer directly. We reproject to EPSG:3826 (TWD97 / TM2 zone 121 — a projected, meter-based CRS for Taiwan) so distances are in meters.
EPSG:4326（經緯度、單位為度）無法直接用於 10 公尺緩衝區運算。我們重新投影至 EPSG:3826（TWD97 / TM2 121 分帶——台灣適用的公尺制投影 CRS），使距離單位為公尺。

In [ ]:
trees_m = trees.to_crs("EPSG:3826")
public_spaces_m = public_spaces.to_crs("EPSG:3826")
print(trees_m.crs.axis_info[0].unit_name)  # expect 'metre'

## Step 3 — Buffer 緩衝區

**Why 為什麼**: we create a 10 m buffer around each tree because we want to ask which public spaces are located within 10 m of a tree — a distance-based research question needs a buffer to become answerable.
我們為每棵樹建立 10 公尺緩衝區，因為我們想問哪些公共空間位於樹木 10 公尺範圍內——以距離為基礎的研究問題，需要透過緩衝區才能被回答。

In [ ]:
tree_buffers = trees_m.copy()
tree_buffers["geometry"] = tree_buffers.buffer(10)
tree_buffers.plot(figsize=(5, 5), color="#a1d99b", alpha=0.6)
plt.title("10m tree buffers 10公尺樹木緩衝區")
plt.show()

## Step 4 — Intersection & Spatial Join 交集與空間 Join

**Why 為什麼**: intersection finds the exact overlap between tree buffers and public spaces. Spatial join then lets us count trees per public space — attaching the *where* to the *what*.
交集運算找出樹木緩衝區與公共空間的精確重疊範圍。接著以空間 join 計算每個公共空間內的樹木數量——將「位置」與「屬性」連結起來。

In [ ]:
# Spatial join: which public space (if any) does each tree buffer's centroid fall near?
# Count trees whose 10m buffer intersects each public space polygon
joined = gpd.sjoin(trees_m, public_spaces_m, how="left", predicate="within")
tree_counts = joined.groupby("name").size().rename("tree_count")

public_spaces_m = public_spaces_m.merge(tree_counts, on="name", how="left")
public_spaces_m["tree_count"] = public_spaces_m["tree_count"].fillna(0)
public_spaces_m

**Expected output 預期輸出**: a table with `name` and `tree_count` columns — Riverside Park should have the most trees given the sample coordinates.
含 `name` 與 `tree_count` 欄位的表格——依範例座標，Riverside Park 應擁有最多樹木。

## Step 5 — Calculate a derived metric 計算衍生指標

Tree density = trees per 1,000 m² of public space area — normalizes for public space size so small and large spaces are comparable.
樹木密度＝每千平方公尺公共空間的樹木數量——依面積標準化，讓大小不同的公共空間可互相比較。

In [ ]:
public_spaces_m["area_m2"] = public_spaces_m.geometry.area
public_spaces_m["tree_density_per_1000m2"] = (
    public_spaces_m["tree_count"] / public_spaces_m["area_m2"] * 1000
)
public_spaces_m[["name", "tree_count", "area_m2", "tree_density_per_1000m2"]]

## Step 6 — Visualize 視覺化

**Why a basemap 為什麼要加底圖**: streets and surrounding context (via `contextily`) turn a plot of colored polygons into a recognizable map — the reader can see where the pattern sits relative to real streets, not just relative to other polygons.
街道與周邊脈絡（透過 `contextily`）能讓色塊多邊形圖變成可辨識的地圖——讀者能看到樣式相對於真實街道的位置，而不只是相對於其他多邊形。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
public_spaces_m.plot(ax=ax, column="tree_density_per_1000m2", cmap="Greens",
                      edgecolor="black", alpha=0.75, legend=True)
trees_m.plot(ax=ax, color="black", markersize=15)

# Basemap, reprojected to match our metric CRS (EPSG:3826)
# 底圖，重新投影至與我們相同的公尺制 CRS（EPSG:3826）
cx.add_basemap(ax, crs=public_spaces_m.crs.to_string(), source=cx.providers.CartoDB.Positron)

ax.set_title("Tree density by public space 各公共空間之樹木密度")
ax.set_axis_off()
plt.show()

## Research interpretation exercise 研究詮釋練習

1. Change the buffer distance from 10 to 20 meters — does the tree count per public space change? Why? 將緩衝區距離從 10 公尺改為 20 公尺——每個公共空間的樹木數量是否改變？為什麼？
2. Which public space has the highest tree density? Apply the What/Where/Why/So-what chain from [Lesson 06](../lessons/06-spatial-insight/). 哪個公共空間樹木密度最高？套用[課程 06](../lessons/06-spatial-insight/)的「是什麼／在哪裡／為什麼／所以呢」推論鏈。
3. Continue to [`04_mini_research.ipynb`](04_mini_research.ipynb) to run this full pipeline on a question of your own.
4. Want the real-data version of this exact analysis? See [`case-studies/01-green-coverage-grid/`](../case-studies/01-green-coverage-grid/) — same method, real Taipei tree data, no registration required. 想看這個分析的真實資料版本？見 [`case-studies/01-green-coverage-grid/`](../case-studies/01-green-coverage-grid/)——方法相同，改用真實台北樹木資料，無需申請帳號。